# Análise de parcelamentos em curso

In [79]:
import pandas as pd


from pathlib import Path
import sys

parent_dir = Path.cwd().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

from utils_ccd import get_connection, get_informacoes_processo

In [74]:
sql_processos = """
SELECT DISTINCT CONCAT(pro.numero_processo, '/', pro.ano_processo) as processo
FROM processo.dbo.Processos pro
	INNER JOIN processo.dbo.Pro_MarcadorProcesso pmp ON pmp.IdProcesso = pro.IdProcesso
	INNER JOIN processo.dbo.Pro_Marcador pm ON pm.IdMarcador = pmp.IdMarcador
WHERE pro.setor_atual = 'CCD'
AND pm.Descricao = 'PARCELAMENTO EM CURSO'

"""
df_processos = pd.read_sql(sql_processos, get_connection())

In [75]:
df_processos

,processo
0,000060/2025
1,000144/2022
2,000402/2025
3,000414/2025
4,000415/2025
...,...
110,200232/2021
111,701069/2011
112,701134/2012
113,701144/2013


In [103]:
sql_infos = f"""
SELECT CONCAT(vai.numero_processo, '/', vai.ano_processo) AS processo,
        pe.SequencialProcessoEvento as evento,
        vai.resumo
FROM processo.dbo.vw_ata_informacao vai
INNER JOIN processo.dbo.Pro_ProcessoEvento pe ON pe.IdInformacao = vai.idInformacao
WHERE CONCAT(vai.numero_processo, '/', vai.ano_processo) IN ({', '.join([f"'{processo}'" for processo in df_processos['processo']])})
ORDER BY 
CONCAT(vai.numero_processo, '/', vai.ano_processo), 
pe.SequencialProcessoEvento
"""
df_infos = pd.read_sql(sql_infos, get_connection())

In [ ]:
df_infos[df_infos.processo == '702561/2012'].resumo

'Capa'

In [123]:
processos_cancelados = []
for p in df_processos.processo:
    _df = df_infos[df_infos.processo == p]
    _df.index[-1]
    max_evento = int(_df.evento.max())
    infos_canceladas = _df[_df.resumo.str.lower().str.contains("ancelamento|arquivam", na=False) & (_df.evento == max_evento)]
    
    if not infos_canceladas.empty:
        processos_cancelados.append(p)
            #infos = _df[_df.resumo.str.lower().str.contains("ancelamento|arquivam", na=False)]

            #if _df.index[-1] in infos.index:
            #    processos_cancelados.append(p)
            #    print(f"Processo {p} tem o último resumo relacionado a cancelamento ou arquivamento.")


In [124]:
processos_cancelados

[]